In [4]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

import os

load_dotenv()

token = os.getenv("GITHUB_TOKEN")
endpoint = "https://models.github.ai/inference"

openai_client = OpenAI(
    base_url=endpoint,
    api_key=token,
)
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [6]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join the program even if it has already begun. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can also start learning and submitting homework without registering, as registration is just a way to gauge interest.'

In [7]:
vs_index.close()